# 05 — OCR Evaluation and Error Slices

Evaluate TrOCR predictions on the OCR training set, generate targeted error subsets,
and benchmark EasyOCR as a baseline for comparison.

In [ ]:
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

In [ ]:
TRAIN_DIR = Path('C:/path/to/private/data/ocr_v4_dataset/train')
MODEL_DIR = Path('C:/path/to/private/models/trocr_plate_model_v2/best_model')
OUTPUT_CSV = Path('trocr_train_eval.csv')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
processor = TrOCRProcessor.from_pretrained(str(MODEL_DIR))
model = VisionEncoderDecoderModel.from_pretrained(str(MODEL_DIR)).to(DEVICE)
model.eval()

results = []
for file_path in tqdm([f for f in TRAIN_DIR.iterdir() if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}], desc='TrOCR eval'):
    gt = file_path.name.split('_')[0].upper()
    pred = ''
    try:
        image = Image.open(file_path).convert('RGB')
        pixel_values = processor(image, return_tensors='pt').pixel_values.to(DEVICE)
        with torch.no_grad():
            ids = model.generate(pixel_values, max_length=16)
        pred = processor.batch_decode(ids, skip_special_tokens=True)[0].upper().replace(' ', '')
    except Exception:
        pass
    results.append({'image_name': file_path.name, 'ground_truth': gt, 'prediction': pred, 'exact_match': pred == gt})

df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)
print('exact_match_accuracy(%) =', round(100 * df['exact_match'].mean() if len(df) else 0, 2))
print('saved =', OUTPUT_CSV)

## Error slice: wrong predictions with ≤2 character differences

In [ ]:
df = pd.read_csv('trocr_train_eval.csv')

rows = []
for _, row in df.iterrows():
    gt = str(row['ground_truth'])
    pred = str(row['prediction'])
    if gt == pred or len(gt) != len(pred):
        continue
    diff = sum(a != b for a, b in zip(gt, pred))
    if diff <= 2:
        rows.append(row)

near_miss_df = pd.DataFrame(rows)
near_miss_df.to_csv('near_miss_errors.csv', index=False)
print('near_miss_count =', len(near_miss_df))

## Error slice: state-code-only mistakes

In [ ]:
df = pd.read_csv('trocr_train_eval.csv')

rows = []
for _, row in df.iterrows():
    gt = str(row['ground_truth'])
    pred = str(row['prediction'])
    if len(gt) != len(pred):
        continue
    if gt[:2] != pred[:2] and gt[2:] == pred[2:]:
        rows.append(row)

state_df = pd.DataFrame(rows)
state_df.to_csv('state_only_errors.csv', index=False)
print('state_only_error_count =', len(state_df))

## EasyOCR baseline (character-level accuracy)

In [ ]:
import os
import re
import cv2
from difflib import SequenceMatcher
import easyocr

IMAGE_DIR = Path('C:/path/to/private/data/ocr_v4_dataset/train')

reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())

def clean_text(text):
    text = text.upper().replace('IND', '')
    return re.sub(r'[^A-Z0-9]', '', text)

files = [f for f in IMAGE_DIR.iterdir() if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}][:1000]
results = []

for file_path in tqdm(files, desc='EasyOCR eval'):
    gt = file_path.name.split('_')[0].upper()
    pred = ''
    try:
        img = cv2.imread(str(file_path))
        img = cv2.resize(img, None, fx=4, fy=4, interpolation=cv2.INTER_CUBIC)
        temp = 'temp_easyocr.jpg'
        cv2.imwrite(temp, img)
        ocr_result = reader.readtext(temp, detail=0)
        pred = clean_text(''.join(ocr_result))
    except Exception:
        pass
    char_acc = SequenceMatcher(None, gt, pred).ratio()
    results.append({'image_name': file_path.name, 'ground_truth': gt, 'prediction': pred,
                    'exact_match': pred == gt, 'char_accuracy': char_acc})

easyocr_df = pd.DataFrame(results)
easyocr_df.to_csv('easyocr_eval.csv', index=False)
print('exact_match_accuracy(%) =', round(100 * easyocr_df['exact_match'].mean(), 2))
print('avg_char_accuracy(%) =', round(100 * easyocr_df['char_accuracy'].mean(), 2))